In [17]:
import torch
from torch.distributions import Distribution, Normal
import math
import numpy as np

In [18]:
# load the parameters, I've only done the wide sample for now
training_parameters = np.load('../CosmoGrid_params.npz').files
param_names = ['Om', 'sigma8', 'w0', 'ns', 'Ob', 'H0', 'bary_Mc', 'bary_nu']
training_parameters = np.array([np.load('../CosmoGrid_params.npz')[par][:1250] for par in param_names]).T

In [32]:
import torch
import math
import numpy as np
from torch.distributions import Distribution

class KDEDistribution(Distribution):
    def __init__(self, samples, bandwidth=0.1, device='cpu'):
        self.device = device
        if isinstance(samples, np.ndarray):
            samples = torch.tensor(samples, dtype=torch.float32, device=device)
        else:
            samples = samples.to(device)

        self.raw_samples = samples  # Unstandardized original samples

        # Compute standardization parameters
        self.sample_mean = samples.mean(0)
        self.sample_std = samples.std(0)

        # Apply standardization
        self.samples = self.standardize(samples)  # shape: [N, D]
        self.N, self.D = self.samples.shape
        self.bandwidth = bandwidth
        super().__init__(validate_args=False)

    def standardize(self, x):
        return (x - self.sample_mean) / self.sample_std

    def destandardize(self, x):
        return x * self.sample_std + self.sample_mean

    def sample(self, sample_shape=torch.Size()):
        if isinstance(sample_shape, tuple):
            n_samples = int(torch.tensor(sample_shape).prod().item())
        else:
            n_samples = int(sample_shape)

        idx = torch.randint(0, self.N, (n_samples,), device=self.device)
        noise = torch.randn(n_samples, self.D, device=self.device) * self.bandwidth
        standardized_samples = self.samples[idx] + noise
        return self.destandardize(standardized_samples)

    def log_prob(self, x):
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32, device=self.device)
        else:
            x = x.to(self.device)

        x_standardized = self.standardize(x)
        x_exp = x_standardized.unsqueeze(-2)          # [..., 1, D]
        s_exp = self.samples.unsqueeze(0)             # [1, N, D]
        dist_sq = ((x_exp - s_exp) ** 2).sum(-1)      # [..., N]
        kernel = torch.exp(-0.5 * dist_sq / self.bandwidth**2)
        norm = (2 * math.pi * self.bandwidth ** 2) ** (self.D / 2)
        return torch.log(kernel.mean(-1) / norm)


In [33]:
'''
import torch
from torch.distributions import Distribution
import math
import numpy as np

# this class can sample from a kde estimate and return log_probs
class KDEDistribution(Distribution):
    def __init__(self, samples, bandwidth=0.1, device='cpu'):
        self.device = device
        if isinstance(samples, np.ndarray):
            samples = torch.tensor(samples, dtype=torch.float32, device=device)
        else:
            samples = samples.to(device)

        self.samples = samples  # shape: [N, D]
        self.N, self.D = self.samples.shape
        self.bandwidth = bandwidth
        super().__init__(validate_args=False)


    def sample(self, sample_shape=torch.Size()):
        if isinstance(sample_shape, tuple):
            n_samples = int(torch.tensor(sample_shape).prod().item())
        else:
            n_samples = int(sample_shape)

        idx = torch.randint(0, self.N, (n_samples,), device=self.device)
        noise = torch.randn(n_samples, self.D, device=self.device) * self.bandwidth
        return self.samples[idx] + noise  # torch.Tensor on self.device

    def log_prob(self, x):
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32, device=self.device)
        else:
            x = x.to(self.device)

        x_exp = x.unsqueeze(-2)             # [..., 1, D]
        s_exp = self.samples.unsqueeze(0)   # [1, N, D]
        dist_sq = ((x_exp - s_exp) ** 2).sum(-1)  # [..., N]
        kernel = torch.exp(-0.5 * dist_sq / self.bandwidth**2)
        norm = (2 * math.pi * self.bandwidth ** 2) ** (self.D / 2)
        return torch.log(kernel.mean(-1) / norm)
'''

"\nimport torch\nfrom torch.distributions import Distribution\nimport math\nimport numpy as np\n\n# this class can sample from a kde estimate and return log_probs\nclass KDEDistribution(Distribution):\n    def __init__(self, samples, bandwidth=0.1, device='cpu'):\n        self.device = device\n        if isinstance(samples, np.ndarray):\n            samples = torch.tensor(samples, dtype=torch.float32, device=device)\n        else:\n            samples = samples.to(device)\n\n        self.samples = samples  # shape: [N, D]\n        self.N, self.D = self.samples.shape\n        self.bandwidth = bandwidth\n        super().__init__(validate_args=False)\n\n\n    def sample(self, sample_shape=torch.Size()):\n        if isinstance(sample_shape, tuple):\n            n_samples = int(torch.tensor(sample_shape).prod().item())\n        else:\n            n_samples = int(sample_shape)\n\n        idx = torch.randint(0, self.N, (n_samples,), device=self.device)\n        noise = torch.randn(n_samples, 

In [34]:
training_parameters.mean(0)

array([ 2.99960625e-01,  8.11987500e-01, -8.98081744e-01,  9.70868750e-01,
        4.50349187e-02,  7.29897062e+01,  1.35124594e+01, -4.20000000e-03])

In [42]:
# we can fit the KDE to the parameters of the simulations
# bandwidth might need some tuning
# if too large, prior is gaussian, if too small prior is noisy
dist = KDEDistribution(training_parameters,bandwidth=.3,device='cpu')

In [43]:
# we can sample by running the following
dist.sample((4,))

tensor([[ 3.2109e-01,  3.9795e-01, -1.3559e+00,  1.0161e+00,  4.5878e-02,
          7.2024e+01,  1.2320e+01, -1.5563e+00],
        [ 1.9657e-01,  8.6727e-01, -7.7467e-01,  9.4681e-01,  4.8859e-02,
          7.0982e+01,  1.3263e+01,  2.4152e+00],
        [ 2.7817e-01,  7.2826e-01, -1.1069e+00,  8.9446e-01,  4.9770e-02,
          7.0537e+01,  1.1746e+01,  1.5756e+00],
        [ 2.2659e-01,  7.9319e-01, -9.3475e-01,  1.0506e+00,  3.2672e-02,
          6.5842e+01,  1.3201e+01,  7.4382e-01]])

In [44]:
# we can look at the prior that we fitted for
import matplotlib.pyplot as plt
from getdist import plots, MCSamples

resample = dist.sample((1000000,)).cpu().numpy()
print('sampled')

strings = np.array(['\Omega_{m}','\sigma_8','w_0','n_s','\Omega_b','H_0','M^0_c','nu'])
samps = MCSamples(samples=resample, names=strings, labels=strings, label='Prior KDE Estimate')
g = plots.get_subplot_plotter()
g.settings.scaling_factor = 1
#g.triangle_plot([samps], strings, filled=True)
g.triangle_plot([samps], strings[[0,1,2,5,3,4]], plot_3d_with_param="nu", filled=False, )
plt.savefig('../Plots/kde_prior_p3_standardized.png')
plt.show()

sampled
Removed no burn in


In [7]:
# quick test if sbi runs
import sys
sys.path.append('./src')
from hyperparameters import hparams
from C3NN import C3NN
from load_data import load_data
from sbi.inference import SNPE_C
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn
import numpy as np
import random
from torch import from_numpy
import torch
import time, datetime
import cloudpickle as cpk


c3nn = C3NN(hparams).to(device='cuda')

training_parameters, training_data = load_data(hparams)

prior = dist

#set up sbi architecture
density_estimator_custom = posterior_nn(model="maf", hidden_features=hparams.flow_hidden_features, num_transforms=hparams.flow_num_transforms, num_blocks=hparams.num_blocks, device=hparams.device, embedding_net=c3nn)
inference = SNPE_C(prior, device=hparams.device, density_estimator=density_estimator_custom)

# joint training for only 2 epochs
density_estimator = inference.append_simulations(training_parameters, training_data).train(learning_rate=hparams.learning_rate, training_batch_size=hparams.flow_training_batch_size, max_num_epochs=2)

/home/k/Kai.Lehman/C3NN_inference/sbi/utils/torchutils.py:27: UserWarning: GPU was selected as a device for training the neural network. Note that we expect **no** significant speed ups in training for the default architectures we provide. Using the GPU will be effective only for large neural networks with operations that are fast on the GPU, e.g., for a CNN or RNN `embedding_net`.
  warnings.warn(
/home/k/Kai.Lehman/C3NN_inference/sbi/utils/user_input_checks.py:435: UserWarning: Mismatch between the device of the data fed to the embedding_net and the device of the embedding_net's weights. Fed data has device 'cpu' vs embedding_net weights have device 'cuda:0'. Automatically switching the embedding_net's device to 'cpu', which could otherwise be done manually using the line `embedding_net.to('cpu')`.
  warnings.warn(


 Training neural network. Epochs trained: 3DANGER!!!! No more deepycopy after training!!!
